# Which samples should enter Ramen's memory? (cache write strategies)

Ramen writes **every** test sample into its memory, filed under CLIP's zero-shot prediction (about 61% correct on CIFAR-10-C). This notebook tests other write rules, mainly: **a sample enters the memory only if zero-shot CLIP and Ramen predict the same class**. When the adapted model agrees with zero-shot, the label is more likely right, so the memory should be cleaner.

The rules are small classes in `src/methods/RamenCache.py`. The method is Ramen in every case; only the write rule changes, chosen with the config key `cache_strategy`:

| strategy | which samples are written | filed under | when |
|---|---|---|---|
| `ramen` | all | zero-shot prediction | before retrieval (exactly Ramen) |
| `all_after` | all | zero-shot prediction | after Ramen's prediction |
| `agree` | only if zero-shot pred == Ramen pred | that class | after Ramen's prediction |
| `adapted_label` | all | Ramen's prediction | after Ramen's prediction |

**Why `all_after` is there.** Agreement can only be checked after Ramen predicts, so `agree` has to write after the prediction. That timing change alone alters Ramen: normally a batch is written first, so each sample also retrieves itself and its batch-mates. `all_after` changes only the timing, so:
- effect of the agree filter = `agree` − `all_after`
- effect of the timing = `all_after` − `ramen`

**Adding a strategy.** In `RamenCache.py`, subclass `CacheStrategy`, implement `select(zs_logits, adapted_logits)` (returns which samples to write and the class each goes under), set `write_after_prediction`, and add it to `STRATEGIES`. Then add its name to the list in section 1.

A full CIFAR-10-C run takes about 10 minutes per strategy. Results are cached in `results/` as `cache_<strategy>`, so re-running a cell is instant.

In [1]:
%load_ext autoreload
%autoreload 2

import os
os.environ.setdefault('CUDA_VISIBLE_DEVICES', '1')  # pick a free GPU; must be set before torch is imported

import numpy as np
import pandas as pd
from ramen_analysis import make_experiment

/home/priyank1/miniconda3/envs/RAMEN/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
DATASET = 'CIFAR10C'  # or 'CIFAR100C', 'ImageNetC5K', 'DomainNet': each uses its paper backbone and cfg/<DATASET>/RamenCache.yaml
exp = make_experiment(DATASET, tta_mode='mixed', batch_size=100, n_per_domain=None, seed=0).load()
acc = exp.evaluate()   # NoAdapt + Ramen (cached)
acc

CIFAR10C: 15 corruptions x 10000 images, 10 classes, model=clip_vitbase32, mode=mixed
NoAdapt: loaded cached results from /home/priyank1/Projects/Ramen/new_experiments/results/CIFAR10C_clip_vitbase32_mixed_nall_bs100_s0_NoAdapt.pkl
Ramen: loaded cached results from /home/priyank1/Projects/Ramen/new_experiments/results/CIFAR10C_clip_vitbase32_mixed_nall_bs100_s0_Ramen.pkl


,NoAdapt,Ramen,delta
gaussian_noise,38.66,60.84,22.18
shot_noise,42.59,63.14,20.55
impulse_noise,42.75,56.27,13.52
defocus_blur,72.61,77.12,4.51
glass_blur,43.91,64.01,20.10
motion_blur,66.68,77.55,10.87
zoom_blur,71.69,79.27,7.58
snow,73.78,80.65,6.87
frost,75.69,80.16,4.47
fog,68.85,77.93,9.08


## 1. Run every strategy
The first check: strategy `ramen` must give exactly Ramen's predictions.

In [3]:
STRATEGIES = ['ramen', 'all_after', 'agree', 'adapted_label']
NAMES = [f'cache_{s}' for s in STRATEGIES]

for s, name in zip(STRATEGIES, NAMES):
    exp.run('RamenCache', name=name, cfg={'cache_strategy': s})

same = (exp.results['cache_ramen'].pred.values == exp.results['Ramen'].pred.values).mean()
print(f"sanity check: strategy 'ramen' gives Ramen's prediction on {same:.4%} of samples")

cache_ramen: loaded cached results from /home/priyank1/Projects/Ramen/new_experiments/results/CIFAR10C_clip_vitbase32_mixed_nall_bs100_s0_cache_ramen.pkl
cache_all_after: loaded cached results from /home/priyank1/Projects/Ramen/new_experiments/results/CIFAR10C_clip_vitbase32_mixed_nall_bs100_s0_cache_all_after.pkl
cache_agree: loaded cached results from /home/priyank1/Projects/Ramen/new_experiments/results/CIFAR10C_clip_vitbase32_mixed_nall_bs100_s0_cache_agree.pkl
cache_adapted_label: loaded cached results from /home/priyank1/Projects/Ramen/new_experiments/results/CIFAR10C_clip_vitbase32_mixed_nall_bs100_s0_cache_adapted_label.pkl
sanity check: strategy 'ramen' gives Ramen's prediction on 100.0000% of samples


## 2. Results
- `accuracy`: average over corruptions (the paper's metric). `vs Ramen`: difference in points. This is seed 0 only: a difference counts only above the noise floor, +0.31 (CIFAR-10-C) / +0.21 (CIFAR-100-C), and needs seeds 0–4 before it is a result.
- `fixed %` / `broken %`: samples zero-shot got wrong and the strategy right / zero-shot right and the strategy wrong.
- `written %`: share of samples that entered the memory.
- `memory label correct %`: of the written samples, the share filed under their true class, i.e. how clean the memory is.
- `same as Ramen %`: share of predictions identical to Ramen's.

In [4]:
def filed_as(name):
    """Class each sample was filed under in the memory (-1 = not written), row by row with exp.results[name]."""
    return np.concatenate(exp.method_stats[name]['filed_as'])


def summary(names):
    base, ramen = exp.results['NoAdapt'], exp.results['Ramen']
    rows = {}
    for name in names:
        r = exp.results[name]
        assert (r.domain_idx.values == base.domain_idx.values).all() and (r.sample_idx.values == base.sample_idx.values).all()
        filed = filed_as(name)
        written = filed >= 0
        avg = r.groupby('domain_idx').correct.mean().mean() * 100
        rows[name] = {
            'accuracy': avg,
            'vs Ramen': avg - acc.loc['average', 'Ramen'],
            'fixed %': (~base.correct.values & r.correct.values).mean() * 100,
            'broken %': (base.correct.values & ~r.correct.values).mean() * 100,
            'written %': written.mean() * 100,
            'memory label correct %': (filed[written] == r.label.values[written]).mean() * 100,
            'same as Ramen %': (r.pred.values == ramen.pred.values).mean() * 100,
        }
    return pd.DataFrame(rows).T.round(2)


summary(NAMES)

,accuracy,vs Ramen,fixed %,broken %,written %,memory label correct %,same as Ramen %
cache_ramen,72.61,0.00,16.92,5.67,100.00,61.37,100.00
cache_all_after,72.01,-0.60,17.90,7.25,100.00,61.37,95.95
cache_agree,71.57,-1.04,17.83,7.62,64.13,83.80,91.93
cache_adapted_label,72.37,-0.24,15.02,4.00,100.00,72.37,84.04


Accuracy per corruption (%).

In [5]:
per_domain = pd.DataFrame({n: exp.results[n].groupby('domain', observed=True).correct.mean() * 100
                           for n in ['NoAdapt', 'Ramen'] + NAMES})
per_domain.loc['average'] = per_domain.mean()
per_domain.round(2)

,NoAdapt,Ramen,cache_ramen,cache_all_after,cache_agree,cache_adapted_label
domain,,,,,,
gaussian_noise,38.66,60.84,60.84,60.42,59.57,57.06
shot_noise,42.59,63.14,63.14,62.86,62.31,59.24
impulse_noise,42.75,56.27,56.27,56.09,57.05,55.35
defocus_blur,72.61,77.12,77.12,76.34,75.40,79.00
glass_blur,43.91,64.01,64.01,64.47,64.35,59.76
motion_blur,66.68,77.55,77.55,77.69,77.13,76.04
zoom_blur,71.69,79.27,79.27,78.53,77.52,80.34
snow,73.78,80.65,80.65,79.42,79.04,81.69
frost,75.69,80.16,80.16,78.02,76.98,82.45


## 3. What does `agree` keep out of the memory?
Per corruption: the share of samples written, and how clean the memory is with `agree` compared with Ramen. Ramen files every sample under its zero-shot prediction, so its memory is exactly as clean as zero-shot is accurate.

In [6]:
r, filed = exp.results['cache_agree'], filed_as('cache_agree')
ramen_filed = filed_as('cache_ramen')
d = pd.DataFrame({'domain': r.domain, 'written': filed >= 0, 'agree right': filed == r.label.values,
                  'ramen right': ramen_filed == r.label.values})
g = d.groupby('domain', observed=True)
t = pd.DataFrame({
    'written %': g.written.mean() * 100,
    'memory label correct %, agree': g['agree right'].sum() / g.written.sum() * 100,
    'memory label correct %, Ramen': g['ramen right'].mean() * 100,
})
t.loc['all'] = [d.written.mean() * 100, d['agree right'].sum() / d.written.sum() * 100, d['ramen right'].mean() * 100]
t.round(2)

,written %,"memory label correct %, agree","memory label correct %, Ramen"
domain,,,
gaussian_noise,39.64,73.69,38.62
shot_noise,43.62,76.41,42.58
impulse_noise,47.63,70.48,42.76
defocus_blur,73.69,87.04,72.64
glass_blur,50.79,74.70,43.93
motion_blur,72.76,84.19,66.69
zoom_blur,73.76,87.41,71.72
snow,74.22,89.61,73.76
frost,74.53,89.37,75.74


## Notes
- Seed 0 only. Before claiming a gain, run the winning strategy at seeds 0–4 (`seed=` in the experiment) and compare seed by seed with the cached Ramen runs.
- A cleaner memory alone isn't enough: RamenClean doubled memory purity on CIFAR-100-C and still lost accuracy, because the effective support shrank. Look at `written %` alongside accuracy.
- To run another dataset, change `DATASET` in the first cell. ImageNet-C and DomainNet read JPEGs from disk, so pass `num_workers=16` to `make_experiment` there.